# Customer Intelligence Platform — Notebook 1: Customer Segmentation
**Portfolio Project | Notebook 2 of 5**

---

## Learning Objectives
1. Build a leak-free preprocessing step (impute → scale) for distance-based clustering
2. Select a cluster count using the Elbow Method **and** Silhouette Analysis together, not either alone
3. Compare K-Means, DBSCAN, and Agglomerative/Hierarchical clustering on the same data and explain *why* they disagree where they disagree
4. Quantify **cluster stability** under bootstrap resampling — a step most tutorials skip, and the difference between a segmentation a business can actually act on and one that just looks nice in a demo
5. Persist the winning segmentation artefact and a cluster-labeled dataset for Notebooks 2 and 3

> **Senior engineer framing:** Anyone can call `KMeans(n_clusters=4).fit(X)`. What separates a production-grade segmentation from a notebook exercise is proving the clusters are (a) not an artifact of one random seed, (b) not better explained by a fundamentally different algorithm's assumptions, and (c) stable enough that "customer 4021 is a Champion" doesn't flip to a different segment every time the model is retrained on slightly different data. This notebook is built around answering those three questions, not just around fitting an algorithm.

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
from pathlib import Path
import itertools

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_theme(style="whitegrid", palette="deep")
RNG_SEED = 42

DATA_PROCESSED_DIR = Path("../data/processed")
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PROCESSED_DIR / "customers.csv")
df.shape

---
## Part 1: Feature Selection & Preprocessing

Clustering uses **only behavioral/usage features** — never `churned` (that's next notebook's supervised target; folding it in here would let the segmentation "cheat" by encoding churn directly), never `customer_id`, and for this first pass, no categorical columns (mixing raw categoricals into a Euclidean-distance algorithm without proper encoding would badly distort distances — that's a Notebook 3 concern once we're inside a `ColumnTransformer`).

Two preprocessing steps are **mandatory** before any distance-based algorithm:
1. **Impute** missing values (`satisfaction_score`, `avg_session_minutes` have some, from Notebook 0) — K-Means/DBSCAN/Agglomerative clustering cannot handle `NaN` at all
2. **Scale** every feature to comparable magnitude — `tenure_months` (~1-60) and `discount_usage_rate` (~0-1) on raw scales would make Euclidean distance almost entirely dominated by whichever feature has the largest numeric range, regardless of its actual importance

### TODO 1 — Build the imputed, scaled feature matrix

**HINT:**
- `feature_cols` should include every numeric behavioral column: `tenure_months`, `monthly_spend`, `total_spend_lifetime`, `recency_days`, `frequency_12m`, `support_tickets_12m`, `discount_usage_rate`, `avg_session_minutes`, `num_products`, `satisfaction_score`
- Use `SimpleImputer(strategy="median")` — median is robust to the skew you saw in `recency_days`/`monthly_spend` in Notebook 0's EDA
- Fit both the imputer and the scaler on `X` (the full dataset) since this is unsupervised — there's no train/test split to worry about leaking across yet (that concept becomes critical again in Notebook 3, where the target *is* used)

In [ ]:
feature_cols = [
    "tenure_months", "monthly_spend", "total_spend_lifetime", "recency_days",
    "frequency_12m", "support_tickets_12m", "discount_usage_rate",
    "avg_session_minutes", "num_products", "satisfaction_score",
]

X = df[feature_cols].copy()

# TODO: impute missing values with the median, then standardize
imputer = ...   # SimpleImputer(strategy="median")
scaler = ...    # StandardScaler()

X_imputed = ...  # imputer.fit_transform(X)
X_scaled = ...   # scaler.fit_transform(X_imputed)

X_scaled.shape

---
## Part 2: K-Means — Choosing K with the Elbow Method and Silhouette Analysis

Recall from Week 3: WCSS/inertia always decreases as $k$ increases (never a reliable stopping rule on its own), while the **silhouette score**

$$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$$

directly measures how well-separated clusters are, and *does* have a genuine maximum. Use both, plotted side by side, and prefer the $k$ where they agree.

### TODO 2 — Elbow + Silhouette scan across k = 2..10

**HINT:** loop `k in range(2, 11)`, fit `KMeans(n_clusters=k, random_state=RNG_SEED, n_init=10)`, collect `.inertia_` and `silhouette_score(X_scaled, labels)` into two lists, then plot both against `k` in a 1×2 subplot.

In [ ]:
k_values = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_values:
    # TODO: fit KMeans(n_clusters=k, random_state=RNG_SEED, n_init=10) on X_scaled
    kmeans = ...
    labels = ...  # kmeans.labels_
    inertias.append(...)          # kmeans.inertia_
    silhouette_scores.append(...)  # silhouette_score(X_scaled, labels)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(list(k_values), inertias, marker="o")
axes[0].set(xlabel="k", ylabel="Inertia (WCSS)", title="Elbow Method")
axes[1].plot(list(k_values), silhouette_scores, marker="o", color="darkorange")
axes[1].set(xlabel="k", ylabel="Silhouette Score", title="Silhouette Analysis")
plt.tight_layout()
plt.show()

**Interpretation:** pick the $k$ where the elbow plot's slope flattens out *and* the silhouette score is at or near its peak. Given this dataset was generated from 4 latent segments, you should see evidence pointing toward `k=4` — but confirm it from the plots yourself rather than hardcoding it because you read this sentence.

### TODO 3 — Fit the final K-Means model and profile each cluster

**HINT:** after fitting with your chosen `k`, attach `cluster_kmeans` back onto a copy of `df`, then `groupby("cluster_kmeans")[feature_cols].mean()` to get a per-cluster profile table — this is how you'd explain the segmentation to a non-technical stakeholder (e.g. "Cluster 2 = high spend, low support tickets, very recent activity").

In [ ]:
CHOSEN_K = ...  # TODO: set based on the elbow/silhouette plots above (e.g. 4)

# TODO: fit the final KMeans model with CHOSEN_K clusters
kmeans_final = ...
df["cluster_kmeans"] = ...  # kmeans_final.labels_

print("Silhouette score:", silhouette_score(X_scaled, df["cluster_kmeans"]))
print(df["cluster_kmeans"].value_counts().sort_index())

# TODO: compute and display the per-cluster feature-mean profile table
cluster_profile = ...  # df.groupby("cluster_kmeans")[feature_cols].mean().round(1)
cluster_profile

### Per-sample silhouette diagram

The aggregate silhouette score can hide a lot — a cluster full of borderline/negative-silhouette points is a red flag even if the *average* score looks acceptable. This diagram (from Week 3) shows the distribution within each cluster, not just the mean.

In [ ]:
sample_silhouette_values = silhouette_samples(X_scaled, df["cluster_kmeans"])

fig, ax = plt.subplots(figsize=(8, 5))
y_lower = 10
for cluster_id in sorted(df["cluster_kmeans"].unique()):
    cluster_values = sample_silhouette_values[df["cluster_kmeans"] == cluster_id]
    cluster_values.sort()
    size = len(cluster_values)
    y_upper = y_lower + size
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values, alpha=0.7)
    ax.text(-0.05, y_lower + size / 2, str(cluster_id))
    y_lower = y_upper + 10

ax.axvline(x=silhouette_score(X_scaled, df["cluster_kmeans"]), color="red", linestyle="--", label="Mean silhouette")
ax.set(xlabel="Silhouette coefficient", ylabel="Cluster", title="Per-sample Silhouette Diagram — K-Means")
ax.legend()
plt.show()

---
## Part 3: DBSCAN — Does a Density-Based View Change the Picture?

K-Means assumes roughly spherical, similarly-sized clusters — an assumption that may or may not hold here. DBSCAN makes no such assumption, but needs `eps` chosen principled-ly via a **k-distance graph**, not guessed.

### TODO 4 — k-distance graph to choose `eps`

**HINT:** fit `NearestNeighbors(n_neighbors=5)` on `X_scaled`, get distances to each point's 5th nearest neighbor, sort them ascending, and plot. The "elbow" in this curve is a principled starting point for `eps`.

In [ ]:
# TODO: build the k-distance graph
k_for_graph = 5
neighbors = ...        # NearestNeighbors(n_neighbors=k_for_graph).fit(X_scaled)
distances, _ = ...      # neighbors.kneighbors(X_scaled)
k_distances = np.sort(distances[:, -1])

plt.figure(figsize=(8, 4.5))
plt.plot(k_distances)
plt.xlabel("Points sorted by distance")
plt.ylabel(f"Distance to {k_for_graph}th nearest neighbor")
plt.title("k-distance graph — look for the elbow")
plt.show()

### TODO 5 — Fit DBSCAN and compare to K-Means

**HINT:** set `eps` to the y-value where the elbow above bends sharply upward (try a value, inspect the noise ratio, adjust), `min_samples=5` is a reasonable starting default (rule of thumb: `min_samples >= n_features + 1`). Compute the number of clusters found (excluding noise, labeled `-1`), the fraction of points labeled noise, and the silhouette score **computed only on non-noise points**.

In [ ]:
DBSCAN_EPS = ...      # TODO: set from the k-distance graph
DBSCAN_MIN_SAMPLES = 5

# TODO: fit DBSCAN(eps=DBSCAN_EPS, min_samples=DBSCAN_MIN_SAMPLES) on X_scaled
dbscan = ...
df["cluster_dbscan"] = ...  # dbscan.labels_

n_clusters_dbscan = len(set(df["cluster_dbscan"])) - (1 if -1 in df["cluster_dbscan"].values else 0)
noise_ratio = (df["cluster_dbscan"] == -1).mean()
non_noise_mask = df["cluster_dbscan"] != -1

print(f"DBSCAN found {n_clusters_dbscan} clusters, {noise_ratio:.1%} labeled as noise")
if n_clusters_dbscan > 1:
    print("Silhouette (non-noise points only):",
          silhouette_score(X_scaled[non_noise_mask], df.loc[non_noise_mask, "cluster_dbscan"]))

---
## Part 4: Agglomerative (Hierarchical) Clustering

A dendrogram gives a visual, multi-resolution view of nested structure that neither K-Means nor DBSCAN can show directly — useful for sanity-checking whether "4 clusters" is really the most natural cut, or whether the data supports a different granularity (e.g. 2 broad clusters that further split into 4).

In [ ]:
# Dendrogram on a random subsample (full 6000 points would be unreadable and slow)
sample_idx = np.random.RandomState(RNG_SEED).choice(len(X_scaled), size=300, replace=False)
linkage_matrix = linkage(X_scaled[sample_idx], method="ward")

plt.figure(figsize=(12, 5))
dendrogram(linkage_matrix, truncate_mode="lastp", p=20)
plt.title("Hierarchical Clustering Dendrogram (Ward linkage, 300-point sample, truncated)")
plt.xlabel("Cluster size (or sample index)")
plt.ylabel("Distance")
plt.show()

### TODO 6 — Fit Agglomerative Clustering and compare all three algorithms

**HINT:** use `AgglomerativeClustering(n_clusters=CHOSEN_K, linkage="ward")` — same `k` as K-Means, for a fair comparison — then compute its silhouette score alongside the other two.

In [ ]:
# TODO: fit AgglomerativeClustering(n_clusters=CHOSEN_K, linkage="ward") on X_scaled
agglomerative = ...
df["cluster_agglomerative"] = ...  # agglomerative.fit_predict(X_scaled)

comparison = pd.DataFrame({
    "algorithm": ["K-Means", "DBSCAN (non-noise)", "Agglomerative"],
    "n_clusters": [
        CHOSEN_K,
        n_clusters_dbscan,
        CHOSEN_K,
    ],
    "silhouette_score": [
        silhouette_score(X_scaled, df["cluster_kmeans"]),
        silhouette_score(X_scaled[non_noise_mask], df.loc[non_noise_mask, "cluster_dbscan"]) if n_clusters_dbscan > 1 else np.nan,
        silhouette_score(X_scaled, df["cluster_agglomerative"]),
    ],
})
comparison

---
## Part 5: Cluster Stability — Would You Get the Same Answer Twice?

A segmentation that reshuffles every time you retrain is operationally useless — marketing can't build a "win back at-risk customers" campaign around a segment definition that isn't stable. We quantify stability by refitting K-Means on several **bootstrap resamples**, predicting labels for the *original* dataset each time, and measuring the **pairwise Adjusted Rand Index (ARI)** between runs — 1.0 means perfect agreement, 0.0 means agreement no better than chance.

### TODO 7 — Bootstrap stability check

**HINT:**
- For `n_bootstrap = 10` iterations: resample `X_scaled` **with replacement** to the same size, fit a fresh `KMeans(n_clusters=CHOSEN_K, random_state=i, n_init=10)` on the resample, then call `.predict(X_scaled)` (not `.labels_`!) to get labels for the *original, full* dataset — this is what makes the label vectors comparable across runs
- Store each run's label vector, then compute `adjusted_rand_score` for every pair of runs using `itertools.combinations`
- Report the mean pairwise ARI — above ~0.8 is generally considered stable for a customer segmentation use case

In [ ]:
n_bootstrap = 10
bootstrap_labels = []
rng = np.random.default_rng(RNG_SEED)

for i in range(n_bootstrap):
    # TODO: resample X_scaled with replacement to the same size as X_scaled
    resample_idx = ...  # rng.integers(0, len(X_scaled), size=len(X_scaled))
    X_resampled = ...   # X_scaled[resample_idx]

    # TODO: fit KMeans on the resample, then predict labels for the ORIGINAL X_scaled
    kmeans_boot = ...
    labels_on_original = ...  # kmeans_boot.predict(X_scaled)
    bootstrap_labels.append(labels_on_original)

pairwise_aris = [
    adjusted_rand_score(a, b)
    for a, b in itertools.combinations(bootstrap_labels, 2)
]
print(f"Mean pairwise ARI across {n_bootstrap} bootstrap resamples: {np.mean(pairwise_aris):.3f}")
print(f"Min pairwise ARI: {np.min(pairwise_aris):.3f}")

---
## Part 6 (Bonus): Validating Against the Ground-Truth Segments

Because Notebook 0 generated this data from known latent segments, we can — uniquely, only because this is synthetic — check how well K-Means recovered the true structure. **In a real deployment you would never have this ground truth**; this section exists purely to validate that the clustering pipeline itself is sound before trusting it on data where you can't check the answer key.

In [ ]:
ground_truth = pd.read_csv(DATA_PROCESSED_DIR / "ground_truth_segments.csv")
df_validated = df.merge(ground_truth, on="customer_id")

# TODO: compute adjusted_rand_score between df_validated["true_segment"] and df_validated["cluster_kmeans"]
ari_vs_ground_truth = ...
print(f"ARI vs. true latent segments: {ari_vs_ground_truth:.3f}")

pd.crosstab(df_validated["true_segment"], df_validated["cluster_kmeans"])

---
## Part 7: Persist the Winning Segmentation

Save the fitted `imputer` + `scaler` + `kmeans_final` (as a dict of artefacts, since clustering isn't a single-estimator `Pipeline` the way supervised learning is) and the cluster-labeled dataset — Notebooks 2 and 3 both build on this output.

In [ ]:
segmentation_artifacts = {
    "imputer": imputer,
    "scaler": scaler,
    "kmeans": kmeans_final,
    "feature_cols": feature_cols,
}
joblib.dump(segmentation_artifacts, MODELS_DIR / "segmentation_artifacts.joblib")

output_cols = ["customer_id"] + feature_cols + ["contract_type", "region", "acquisition_channel", "payment_method", "churned", "cluster_kmeans"]
df[output_cols].rename(columns={"cluster_kmeans": "segment"}).to_csv(
    DATA_PROCESSED_DIR / "customers_with_segments.csv", index=False
)
print("Saved segmentation_artifacts.joblib and customers_with_segments.csv")

---
## Senior Engineer Notes & Best Practices

1. **Never let the supervised target leak into an unsupervised step.** `churned` was deliberately excluded from clustering — segmentation should describe *behavior*, and then you separately test whether that behavioral grouping happens to correlate with churn (a legitimate, interesting finding) rather than baking churn into the segment definition itself (circular).
2. **Elbow + silhouette together, never either alone** — inertia always decreases with $k$; silhouette can have a spurious peak at $k=2$ for some datasets. Cross-check both, and sanity-check the resulting cluster sizes aren't absurdly imbalanced.
3. **Algorithm disagreement is information, not noise.** If DBSCAN finds very different structure than K-Means, that tells you something concrete about whether your data is convex-cluster-shaped — don't just pick whichever algorithm gives the highest silhouette score without understanding why.
4. **Stability testing is the step that separates a real segmentation from a demo.** A silhouette score of 0.55 on one fit means nothing if a bootstrap resample gives you a completely different partition — always check both.
5. **Ground-truth validation is a luxury of synthetic/labeled data — use it to validate your *pipeline*, not as a substitute for stability testing on real, unlabeled data**, where it won't be available.
6. **Persist the full preprocessing state (imputer + scaler), not just the fitted clusterer.** A new customer's raw features must go through *exactly* the same imputation/scaling before `.predict()` — this is the same train/serve-skew concern you'll see formalized as a `Pipeline` in Notebook 3.

## Key Takeaways
- A trustworthy segmentation requires: leak-free features, a principled $k$ selection (elbow + silhouette), cross-algorithm comparison, and quantified stability under resampling
- K-Means, DBSCAN, and Agglomerative clustering encode different geometric assumptions — running only one and trusting the result is a common shortcut that production segmentations can't afford
- The output of this notebook (`customers_with_segments.csv`, `segmentation_artifacts.joblib`) is a real dependency for Notebooks 2 and 3, not just a standalone exercise